# Import delle librerie e caricamento dati

In [26]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
import pickle
import re
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import ast
from collections import Counter

In [3]:
df_campione = pd.read_csv('campione_dev_bilanciato.csv')
print(f"Claim caricate: {len(df_campione):,}")
print(df_campione['label'].value_counts().to_string())

with open('corpus_fever_arricchito.pkl', 'rb') as f:
    corpus_arricchito = pickle.load(f)
print(f"\nDocumenti corpus: {len(corpus_arricchito):,}")

Claim caricate: 1,500
label
SUPPORTS           500
NOT ENOUGH INFO    500
REFUTES            500

Documenti corpus: 14,364


Si estraggono 10 claim e 5 frasi di documenti come testi di test

In [4]:
claim_esempio = df_campione['testo_claim'].head(10).tolist()
doc_esempio = [
    doc['text'][:300]
    for doc in list(corpus_arricchito.values())[:5]
    if doc['text']
]

print("Claim di test:")
for i, c in enumerate(claim_esempio):
    print(f"  {i+1}. {c}")

print("\nFrasi corpus di test:")
for i, d in enumerate(doc_esempio):
    print(f"  {i+1}. {d[:100]}")

Claim di test:
  1. Alberto Contador was stripped of his wins.
  2. In golf, players hit balls into holes.
  3. The Monster is a work of music by an American rapper.
  4. Speech recognition incorporates knowledge and research into multiple fields.
  5. The Sterile Cuckoo's adaptation was done by Alvin Sargent.
  6. Ding Yanyuhang is male.
  7. Emmanuel Macron was employed as an investment banker.
  8. Part of Hindu Kush is in Pakistan.
  9. There is a screenwriter by the name of Fred Armisen.
  10. Adidas designs things that you can wear.

Frasi corpus di test:
  1. 19 Kids and Counting -LRB- formerly 17 Kids and Counting and 18 Kids and Counting -RRB- was an Ameri
  2. The 1985 -- 86 NBA season was the 40th season of the National Basketball Association . The season en
  3. The 2015 IIFA Awards , officially known as the 16th International Indian Film Academy Awards ceremon
  4. 12 Play is the debut studio album by American R&B and soul musician R. Kelly ; it was released on No


# Tokenizzazione

Confronto word_tokenize vs split

In [5]:
print("Confronto tokenizzatori")
testi_confronto = [
    "Nikolaj Coster-Waldau worked with the Fox Broadcasting Company.",
    "It's a well-known fact that the U.S. has 50 states.",
    "The film grossed $1.2 billion world-wide."
]
for testo in testi_confronto:
    print(f"\nTesto: {testo}")
    print(f"  split():          {testo.split()}")
    print(f"  word_tokenize():  {word_tokenize(testo)}")

Confronto tokenizzatori

Testo: Nikolaj Coster-Waldau worked with the Fox Broadcasting Company.
  split():          ['Nikolaj', 'Coster-Waldau', 'worked', 'with', 'the', 'Fox', 'Broadcasting', 'Company.']
  word_tokenize():  ['Nikolaj', 'Coster-Waldau', 'worked', 'with', 'the', 'Fox', 'Broadcasting', 'Company', '.']

Testo: It's a well-known fact that the U.S. has 50 states.
  split():          ["It's", 'a', 'well-known', 'fact', 'that', 'the', 'U.S.', 'has', '50', 'states.']
  word_tokenize():  ['It', "'s", 'a', 'well-known', 'fact', 'that', 'the', 'U.S.', 'has', '50', 'states', '.']

Testo: The film grossed $1.2 billion world-wide.
  split():          ['The', 'film', 'grossed', '$1.2', 'billion', 'world-wide.']
  word_tokenize():  ['The', 'film', 'grossed', '$', '1.2', 'billion', 'world-wide', '.']


Si divide il testo in una lista di token

In [6]:
testo_prova = claim_esempio[0]
print(f"Testo originale:\n  {testo_prova}")

token_grezzi = word_tokenize(testo_prova)
print(f"Token (word_tokenize NLTK):\n  {token_grezzi}")
print(f"{len(token_grezzi)} token")

Testo originale:
  Alberto Contador was stripped of his wins.
Token (word_tokenize NLTK):
  ['Alberto', 'Contador', 'was', 'stripped', 'of', 'his', 'wins', '.']
8 token


# Lowercase e rimozione punteggiatura

In [7]:
token_minuscolo = [t.lower() for t in token_grezzi]
print(f"Dopo lowercase:\n  {token_minuscolo}")

Dopo lowercase:
  ['alberto', 'contador', 'was', 'stripped', 'of', 'his', 'wins', '.']


In [8]:
token_puliti = [re.sub(r'[^a-z0-9]', '', t) for t in token_minuscolo]
token_puliti = [t for t in token_puliti if t]
print(f"Dopo rimozione punteggiatura:\n  {token_puliti}")

rimossi = set(token_minuscolo) - set(token_puliti)
print(f"Token rimossi: {rimossi}")

Dopo rimozione punteggiatura:
  ['alberto', 'contador', 'was', 'stripped', 'of', 'his', 'wins']
Token rimossi: {'.'}


# Rimozione stopwords

In [9]:
lista_stopwords = set(stopwords.words('english'))
print(f"Stopwords NLTK (inglese): {len(lista_stopwords)} parole")
print(f"Esempi: {sorted(lista_stopwords)[:20]}")

Stopwords NLTK (inglese): 198 parole
Esempi: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been']


In [10]:
token_senza_stopwords = [t for t in token_puliti if t not in lista_stopwords]
print(f"Prima della rimozione ({len(token_puliti)} token):\n  {token_puliti}")
print()
print(f"Dopo la rimozione ({len(token_senza_stopwords)} token):\n  {token_senza_stopwords}")
print()
rimossi_sw = set(token_puliti) - set(token_senza_stopwords)
print(f"Stopwords rimosse: {rimossi_sw}")

Prima della rimozione (7 token):
  ['alberto', 'contador', 'was', 'stripped', 'of', 'his', 'wins']

Dopo la rimozione (4 token):
  ['alberto', 'contador', 'stripped', 'wins']

Stopwords rimosse: {'his', 'was', 'of'}


In [12]:
claim_con_numeri = [
    c for c in claim_esempio
    if any(char.isdigit() for char in c)
][:3]

print("VERIFICA CLAIM CON NUMERI/ANNI")
print("=" * 55)
for claim in claim_con_numeri:
    tk = word_tokenize(claim.lower())
    tk = [re.sub(r'[^a-z0-9]', '', t) for t in tk if t]
    tk = [t for t in tk if t]
    tk_no_sw = [t for t in tk if t not in lista_stopwords]
    print(f"\nClaim: {claim}")
    print(f"Token: {tk_no_sw}")

VERIFICA CLAIM CON NUMERI/ANNI


# Stemming Porter

Si riduce ogni parola alla sua radice

In [13]:
stemmer = PorterStemmer()

token_stemmati = [stemmer.stem(t) for t in token_senza_stopwords]
print(f"Prima dello stemming:\n  {token_senza_stopwords}")
print()
print(f"Dopo lo stemming:\n  {token_stemmati}")
print()

print("Trasformazioni:")
for originale, stemmato in zip(token_senza_stopwords, token_stemmati):
    if originale != stemmato:
        print(f"  {originale:<20} → {stemmato}")

Prima dello stemming:
  ['alberto', 'contador', 'stripped', 'wins']

Dopo lo stemming:
  ['alberto', 'contador', 'strip', 'win']

Trasformazioni:
  stripped             → strip
  wins                 → win


Controllo su parole tipiche del dominio fever

In [14]:
parole_dominio = [
    'broadcasting', 'worked', 'released', 'directed',
    'American', 'starred', 'filmed', 'produced',
    'nationality', 'organization', 'political'
]
print("STEMMING SU PAROLE TIPICHE DEL DOMINIO")
print("=" * 45)
for parola in parole_dominio:
    print(f"  {parola:<20} → {stemmer.stem(parola.lower())}")

STEMMING SU PAROLE TIPICHE DEL DOMINIO
  broadcasting         → broadcast
  worked               → work
  released             → releas
  directed             → direct
  American             → american
  starred              → star
  filmed               → film
  produced             → produc
  nationality          → nation
  organization         → organ
  political            → polit


# Pipeline completa

In [16]:
ANNOTAZIONI_FEVER = re.compile(r'-LRB-|-RRB-|-LSB-|-RSB-|-LCB-|-RCB-')

In [17]:
def elabora_testo(testo: str) -> list[str]:
    if not testo or not testo.strip():
        return []

    # 0. Rimozione annotazioni FEVER
    testo = ANNOTAZIONI_FEVER.sub(' ', testo)

    # 1. Tokenizzazione
    token = word_tokenize(testo)

    # 2. Lowercase
    token = [t.lower() for t in token]

    # 3. Rimozione punteggiatura
    token = [re.sub(r'[^a-z0-9]', '', t) for t in token]
    token = [t for t in token if t]

    # 4. Rimozione stopwords
    token = [t for t in token if t not in lista_stopwords]

    # 5. Stemming
    token = [stemmer.stem(t) for t in token]

    return token

Esempi applicazione pipeline

In [18]:
testo_prova = "Nikolaj Coster-Waldau worked with the Fox Broadcasting Company."
risultato = elabora_testo(testo_prova)
print(f"Input:    {testo_prova}")
print(f"Output:   {risultato}")

Input:    Nikolaj Coster-Waldau worked with the Fox Broadcasting Company.
Output:   ['nikolaj', 'costerwaldau', 'work', 'fox', 'broadcast', 'compani']


In [19]:
print("Pipeline completa su 10 claim")
for i, claim in enumerate(claim_esempio):
    token = elabora_testo(claim)
    print(f"\n[{i+1}] {claim}")
    print(f"     → {token}")

Pipeline completa su 10 claim

[1] Alberto Contador was stripped of his wins.
     → ['alberto', 'contador', 'strip', 'win']

[2] In golf, players hit balls into holes.
     → ['golf', 'player', 'hit', 'ball', 'hole']

[3] The Monster is a work of music by an American rapper.
     → ['monster', 'work', 'music', 'american', 'rapper']

[4] Speech recognition incorporates knowledge and research into multiple fields.
     → ['speech', 'recognit', 'incorpor', 'knowledg', 'research', 'multipl', 'field']

[5] The Sterile Cuckoo's adaptation was done by Alvin Sargent.
     → ['steril', 'cuckoo', 'adapt', 'done', 'alvin', 'sargent']

[6] Ding Yanyuhang is male.
     → ['ding', 'yanyuhang', 'male']

[7] Emmanuel Macron was employed as an investment banker.
     → ['emmanuel', 'macron', 'employ', 'invest', 'banker']

[8] Part of Hindu Kush is in Pakistan.
     → ['part', 'hindu', 'kush', 'pakistan']

[9] There is a screenwriter by the name of Fred Armisen.
     → ['screenwrit', 'name', 'fred', 'a

In [20]:
print("Pipeline completa su 5 frasi del corpus")
for i, testo in enumerate(doc_esempio):
    token = elabora_testo(testo)
    print(f"\n[{i+1}] {testo[:100]}")
    print(f"     → {token[:15]}  {'...' if len(token) > 15 else ''}")

Pipeline completa su 5 frasi del corpus

[1] 19 Kids and Counting -LRB- formerly 17 Kids and Counting and 18 Kids and Counting -RRB- was an Ameri
     → ['19', 'kid', 'count', 'formerli', '17', 'kid', 'count', '18', 'kid', 'count', 'american', 'realiti', 'televis', 'show', 'air']  ...

[2] The 1985 -- 86 NBA season was the 40th season of the National Basketball Association . The season en
     → ['1985', '86', 'nba', 'season', '40th', 'season', 'nation', 'basketbal', 'associ', 'season', 'end', 'boston', 'celtic', 'win', 'third']  ...

[3] The 2015 IIFA Awards , officially known as the 16th International Indian Film Academy Awards ceremon
     → ['2015', 'iifa', 'award', 'offici', 'known', '16th', 'intern', 'indian', 'film', 'academi', 'award', 'ceremoni', 'present', 'intern', 'indian']  ...

[4] 12 Play is the debut studio album by American R&B and soul musician R. Kelly ; it was released on No
     → ['12', 'play', 'debut', 'studio', 'album', 'american', 'r', 'b', 'soul', 'musician', 

# Verifica matching claim - documento

In [22]:
def estrai_frase(lines: str, sentence_id: int) -> str:
    if not lines:
        return ""
    for riga in lines.split('\n'):
        riga = riga.strip()
        if not riga:
            continue
        parti = riga.split('\t', 1)
        if len(parti) == 2:
            try:
                if int(parti[0]) == sentence_id:
                    return parti[1].strip()
            except ValueError:
                continue
    return ""

In [23]:
def converti_lista(valore):
    try:
        return ast.literal_eval(valore)
    except:
        return []

In [24]:
df_campione['url_documenti'] = df_campione['url_documenti'].apply(converti_lista)
df_campione['sentence_ids']  = df_campione['sentence_ids'].apply(converti_lista)
print("Colonne caricate correttamente.")

Colonne caricate correttamente.


In [25]:
verificate = 0
for _, riga in df_campione.iterrows():
    if riga['label'] == 'NOT ENOUGH INFO':
        continue
    if not riga['url_documenti']:
        continue

    url = riga['url_documenti'][0]
    sid = riga['sentence_ids'][0] if riga['sentence_ids'] else 0

    if url not in corpus_arricchito:
        continue

    testo_evidenza = estrai_frase(corpus_arricchito[url]['lines'], sid)
    if not testo_evidenza:
        continue

    token_claim = elabora_testo(riga['testo_claim'])
    token_evidenza = elabora_testo(testo_evidenza)

    token_comuni = set(token_claim) & set(token_evidenza)
    overlap_pct = len(token_comuni) / len(set(token_claim)) * 100 if token_claim else 0

    print(f"\n[{riga['label']}]")
    print(f"  Claim:    {riga['testo_claim']}")
    print(f"  Evidenza: {testo_evidenza[:100]}")
    print(f"  Token claim:    {token_claim}")
    print(f"  Token evidenza: {token_evidenza[:10]}...")
    print(f"  Token comuni:   {token_comuni}")
    print(f"  Overlap:        {overlap_pct:.0f}%")

    verificate += 1
    if verificate >= 5:
        break


[SUPPORTS]
  Claim:    Alberto Contador was stripped of his wins.
  Evidenza: He initially also won the 2010 Tour de France with this team , and the 2011 Giro d'Italia with team 
  Token claim:    ['alberto', 'contador', 'strip', 'win']
  Token evidenza: ['initi', 'also', '2010', 'tour', 'de', 'franc', 'team', '2011', 'giro', 'ditalia']...
  Token comuni:   {'strip'}
  Overlap:        25%

[SUPPORTS]
  Claim:    Speech recognition incorporates knowledge and research into multiple fields.
  Evidenza: It incorporates knowledge and research in the linguistics , computer science , and electrical engine
  Token claim:    ['speech', 'recognit', 'incorpor', 'knowledg', 'research', 'multipl', 'field']
  Token evidenza: ['incorpor', 'knowledg', 'research', 'linguist', 'comput', 'scienc', 'electr', 'engin', 'field', 'linguist']...
  Token comuni:   {'field', 'incorpor', 'research', 'knowledg'}
  Overlap:        57%

[SUPPORTS]
  Claim:    The Sterile Cuckoo's adaptation was done by Alvin Sargen

In [27]:
vocabolario_grezzo = Counter()
vocabolario_preprocessato = Counter()

print("Analisi vocabolario su tutto il campione")
for claim in df_campione['testo_claim']:
    for parola in claim.lower().split():
        vocabolario_grezzo[parola] += 1
    for token in elabora_testo(claim):
        vocabolario_preprocessato[token] += 1

print(f"Vocabolario GREZZO:          {len(vocabolario_grezzo):,} parole uniche")
print(f"Vocabolario PREPROCESSATO:   {len(vocabolario_preprocessato):,} parole uniche")
riduzione = (1 - len(vocabolario_preprocessato) / len(vocabolario_grezzo)) * 100
print(f"Riduzione:                   {riduzione:.1f}%")

Analisi vocabolario su tutto il campione
Vocabolario GREZZO:          4,032 parole uniche
Vocabolario PREPROCESSATO:   2,985 parole uniche
Riduzione:                   26.0%
